In [1]:
import MDAnalysis as mda
from numpy import *
import os
from pylab import *
import MDAnalysis.analysis.distances
import MDAnalysis.analysis.rms
from MDAnalysis.analysis import align
import glob
#import umap
import scipy.stats
import sklearn
import sklearn.decomposition
import sklearn.preprocessing
import pandas as pd
import seaborn as sns
from MDAnalysis.analysis.hydrogenbonds.hbond_analysis import HydrogenBondAnalysis as HBA

In [2]:
import os

########################################################
#############      FOR NOW EQPOINT IS 0   ##############
########################################################
EQPOINT=0

systemFolders = sorted(glob.glob("huNumbering/*t5a*/"))

systemgros=[]
systemtprs=[]
systemtrjs=[]
for i in range(len(systemFolders)):
    systemgros.append(sorted(glob.glob(systemFolders[i]+"*.gro")))
    systemtprs.append(sorted(glob.glob(systemFolders[i]+"*.tpr")))
    systemtrjs.append(sorted(glob.glob(systemFolders[i]+"*.xtc")))


    
    
threeColor=["#FE6100","#332288","#882255"]
colourScheme= threeColor
system_names = ["rhT5A","T5A","T5AR332P"]
systems=[]
for i in range(len(systemgros)):
    sub=[]
    for j in range(len(systemgros[i])):
        # When using TPRs, residues are indexed from 1; so we need to add in the first residue, 1 - 1 + first resid=first resid
        #firstres = mda.Universe(systemgros[i][j]).residues.resids[0]-1
        tu = mda.Universe(systemgros[i][j],systemtrjs[i][j])
        #tu.residues.resids +=firstres
                          
        sub.append(tu)
        
    systems.append(sub)


v123s=[]
v123strings=[]
combinedLoopString="resid 324:349 or resid 376:400 or resid 414:430"
#system_loop_strings = [rhloopstring,huloopstring,huloopstring]
for i in range(len(systems)):
    sub = []
    sub2 = []

    for j in range(len(systems[i])):
        sub.append(systems[i][j].select_atoms("protein and "+combinedLoopString))
        sub2.append("protein and "+combinedLoopString)

        
    v123s.append(sub)        
    v123strings.append(sub2)    

C:\Users\Liam\anaconda3\lib\site-packages\MDAnalysis\topology\base.py:203: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  residx = np.zeros_like(criteria[0], dtype=np.int)
C:\Users\Liam\anaconda3\lib\site-packages\MDAnalysis\core\selection.py:640: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: h

In [3]:
# Ok let's do PCA
def getPairwiseDists(systems,datasets,selection_strings,stride = 10,EQPOINT=250):
    
    alldists=[]
    for i in range(len(datasets)):
        subdists=[]
        for j in range(len(datasets[i])):
            distances=[]
            for k in range(int(EQPOINT/stride),int(len(systems[i][j].trajectory)/stride)):
                systems[i][j].trajectory[k*stride]
                distances.append(MDAnalysis.analysis.distances.self_distance_array(datasets[i][j].select_atoms("name CA").positions))
            subdists.append(distances)
        alldists.append(subdists)
    return alldists

dists = getPairwiseDists(systems,v123s,v123strings,stride = 10,EQPOINT=EQPOINT)


In [4]:
dists

[[[array([ 3.92130168,  6.46604505, 10.03112759, ...,  3.85079361,
           6.84138517,  3.92595709]),
   array([3.94516078, 6.62024022, 9.10240353, ..., 3.83719963, 6.61855992,
          3.84650631]),
   array([3.85494211, 6.87282799, 9.58973141, ..., 3.84199816, 6.8924479 ,
          3.77010459]),
   array([3.97221795, 6.64401959, 9.28825828, ..., 3.83433432, 6.44896032,
          3.880148  ]),
   array([3.79812658, 6.59131024, 9.73598203, ..., 3.91748478, 6.79135535,
          3.84236953]),
   array([ 3.83384982,  6.55964646, 10.29307129, ...,  3.89462839,
           6.60313885,  3.77492887]),
   array([3.8145365 , 6.52914891, 8.88264436, ..., 3.87387095, 6.62098364,
          3.84553673]),
   array([ 3.90418289,  6.702611  , 10.32491351, ...,  3.8349033 ,
           6.4113958 ,  3.88989392]),
   array([3.90621534, 6.74660191, 6.69849714, ..., 3.84010351, 6.71029787,
          3.82645061]),
   array([3.79760618, 6.80620737, 6.94507749, ..., 3.82719504, 6.57706694,
          3.6505